# BirdCLEF+ 2026 — MHTAN: Multi-Head Temporal Attention Network

## LLM-Guided Hyperparameter Optimization with neuropt

### Key Contributions
1. **Multi-Head Temporal Attention** over Perch v2 embeddings — transformer-based temporal modeling for 12-window bioacoustic sequences
2. **Learnable Perch Residual Gate** — architecture-level knowledge distillation via sigmoid-gated blend with foundation model logits
3. **neuropt Integration** — LLM-powered hyperparameter search where Claude analyzes training curves to propose next experiments
4. **Bayesian Site/Hour Priors** — metadata-driven species probability estimation with hierarchical blending

### Architecture
```
60s soundscape → Perch v2 → 12 × 1536-dim embeddings
    ↓
Linear projection (1536 → d_model) + Positional Encoding
    ↓
N × [Pre-Norm Multi-Head Self-Attention + FFN] blocks
    ↓
Classification Head → species logits
    ↓
Learnable gate: (1-α)·MHTAN + α·Perch logits
    ↓
Prior fusion + temporal smoothing → submission
```

## Setup
Install dependencies and configure mode.

In [ ]:
import subprocess, sys, os, time
from pathlib import Path

_WALL_START = time.time()  # Track wall time for safety

MODE = "submit"
assert MODE in {"optimize", "submit"}

if MODE == "optimize":
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
        if api_key:
            os.environ["ANTHROPIC_API_KEY"] = api_key
    except Exception:
        pass

# Install TF 2.20 from local wheels (fastest) or fallback
_WHL = Path('/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel')
_WHL2 = Path('/kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels')  # alternate source
if _WHL.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL / 'tensorboard-2.20.0-py3-none-any.whl')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL / 'tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')], check=True)
elif _WHL2.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL2 / 'tensorboard-2.20.0-py3-none-any.whl')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL2 / 'tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')], check=True)
else:
    try:
        import tensorflow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
            'tensorflow==2.20.0', 'tensorboard==2.20.0'], check=True)

if MODE == "optimize":
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuropt', 'anthropic'], check=True)

print(f"Setup done in {time.time()-_WALL_START:.0f}s | MODE={MODE}")


## Imports & Global Config

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc, json, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

# CPU threading optimization
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(os.cpu_count() or 4)

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12
DEVICE = torch.device("cpu")

print(f"TF {tf.__version__} | PyTorch {torch.__version__} | "
      f"threads: inter={tf.config.threading.get_inter_op_parallelism_threads()}, "
      f"intra={tf.config.threading.get_intra_op_parallelism_threads()}")


## Frozen Best Config
These values are populated after running neuropt optimization.  
In `optimize` mode, they serve as defaults; neuropt will search for better ones.  
In `submit` mode, these are used directly.

In [ ]:
FROZEN_BEST_CONFIG = {
    # Architecture (neuropt-discovered, 50 evals, best AUC=0.9410)
    "d_model": 208, "n_heads": 4, "n_layers": 2, "d_ff_mult": 4,
    "dropout": 0.089, "use_perch_residual": True,
    # Training
    "lr": 1.82e-3, "weight_decay": 3.8e-4, "n_epochs": 111,
    "pos_weight_cap": 20.5, "label_smoothing": 0.059,
    # Fusion (also neuropt-discovered)
    "ensemble_weight_attn": 0.558, "temperature": 1.005,
    "lambda_event": 0.4, "lambda_texture": 1.0,
    "lambda_proxy_texture": 0.8, "smooth_texture": 0.35, "smooth_event": 0.15,
}

RUN_CFG = {
    "batch_files": 32,
    "proxy_reduce": "max",
    "dryrun_n_files": 50 if MODE == "optimize" else 20,
    "require_full_cache_in_submit": True,
    "cache_input_dirs": [
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        Path("/kaggle/input/perch-meta"),
    ],
    "full_cache_work_dir": Path("/kaggle/working/perch_cache"),
}
RUN_CFG["full_cache_work_dir"].mkdir(parents=True, exist_ok=True)


## Data Loading
Load competition data, parse labels, build truth matrices.

In [ ]:
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)

taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {
            "file_id": None, "site": None, "date": pd.NaT,
            "time_utc": None, "hour_utc": -1, "month": -1,
        }
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {
        "file_id": file_id, "site": site, "date": dt,
        "time_utc": hms, "hour_utc": int(hms[:2]),
        "month": int(dt.month) if pd.notna(dt) else -1,
    }

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)

meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)

for i, labels in enumerate(sc_clean["label_list"]):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print("sc_clean:", sc_clean.shape)
print("Y_SC:", Y_SC.shape, Y_SC.dtype)
print("Full files:", len(full_files))
print("Trusted full windows:", len(full_truth))
print("Active classes in full windows:", int((Y_FULL_TRUTH.sum(axis=0) > 0).sum()))

## Perch v2 Model & Species Mapping
Load the Perch foundation model and build mapping from BirdCLEF labels to Perch classifier indices.

In [ ]:
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

taxonomy_lookup = taxonomy.copy()
taxonomy_lookup["scientific_name_lookup"] = taxonomy_lookup["scientific_name"]

bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})

mapping = taxonomy_lookup.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup", how="left"
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}

ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA], dtype=np.int32
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA], dtype=np.int32
)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32
)

# Genus-level proxies for unmapped species
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "genus": genus,
            "bc_indices": hits["bc_index"].astype(int).tolist(),
        }

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys() if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])

selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Mapped classes: {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Unmapped classes: {(~MAPPED_MASK).sum()}")
print(f"Proxy targets: {len(SELECTED_PROXY_TARGETS)}")

## Metrics & Utilities

In [ ]:
def macro_auc_skip_empty(y_true, y_score):
    """Competition metric: macro AUC skipping classes with no positives."""
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def smooth_cols_fixed12(scores, cols, alpha=0.35):
    """Temporal smoothing for continuous species (texture): neighbor averaging."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s

def smooth_events_fixed12(scores, cols, alpha=0.15):
    """Temporal smoothing for event birds: soft max-pool preserves transient calls."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

## Perch Inference Engine
Extract Perch v2 embeddings and mapped logits from 60-second soundscapes.

In [ ]:
def read_soundscape_60s(path):
    """Load 60-second soundscape at 32kHz mono."""
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce="max",
                                subsample=False):
    """Run Perch v2 on soundscapes.
    
    If subsample=True, only process even-indexed windows (0,2,4,6,8,10)
    and interpolate odd windows from neighbors. Cuts inference time ~50%.
    """
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS  # always 12 output rows per file

    if subsample:
        COMPUTE_IDX = [0, 2, 4, 6, 8, 10]  # windows we actually run Perch on
        N_COMPUTE = len(COMPUTE_IDX)
    else:
        COMPUTE_IDX = list(range(N_WINDOWS))
        N_COMPUTE = N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_file = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)

        # Load audio and extract only the windows we need
        x = np.empty((batch_n * N_COMPUTE, WINDOW_SAMPLES), dtype=np.float32)
        batch_meta = []
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_60s(path)
            all_windows = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
            x[x_pos:x_pos + N_COMPUTE] = all_windows[COMPUTE_IDX]
            batch_meta.append({
                "stem": path.stem,
                "name": path.name,
                "meta": parse_soundscape_filename(path.name),
            })
            x_pos += N_COMPUTE

        # Run Perch on selected windows only
        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32, copy=False)
        emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

        # Fill in results for each file
        for fi, bm in enumerate(batch_meta):
            row_base = (write_file + fi) * N_WINDOWS
            perch_base = fi * N_COMPUTE
            stem = bm["stem"]
            meta_info = bm["meta"]

            # Fill metadata for all 12 windows
            row_ids[row_base:row_base + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[row_base:row_base + N_WINDOWS] = bm["name"]
            sites[row_base:row_base + N_WINDOWS] = meta_info["site"]
            hours[row_base:row_base + N_WINDOWS] = int(meta_info["hour_utc"])

            # Get Perch outputs for computed windows
            batch_logits = logits[perch_base:perch_base + N_COMPUTE]  # (N_COMPUTE, 10806)
            batch_emb = emb[perch_base:perch_base + N_COMPUTE]        # (N_COMPUTE, 1536)

            # Map to competition labels for computed windows
            computed_scores = np.zeros((N_COMPUTE, N_CLASSES), dtype=np.float32)
            computed_scores[:, MAPPED_POS] = batch_logits[:, MAPPED_BC_INDICES]
            for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
                sub = batch_logits[:, bc_idx_arr]
                if proxy_reduce == "max":
                    computed_scores[:, pos] = sub.max(axis=1).astype(np.float32)
                else:
                    computed_scores[:, pos] = sub.mean(axis=1).astype(np.float32)

            if subsample:
                # Place computed windows and interpolate missing ones
                full_scores = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)
                full_emb = np.zeros((N_WINDOWS, 1536), dtype=np.float32)
                for ci, wi in enumerate(COMPUTE_IDX):
                    full_scores[wi] = computed_scores[ci]
                    full_emb[wi] = batch_emb[ci]
                # Interpolate odd windows as average of neighbors
                for wi in [1, 3, 5, 7, 9, 11]:
                    prev_w = wi - 1  # always even, always computed
                    next_w = min(wi + 1, 10)  # clamp to last computed
                    full_scores[wi] = 0.5 * (full_scores[prev_w] + full_scores[next_w])
                    full_emb[wi] = 0.5 * (full_emb[prev_w] + full_emb[next_w])
                scores[row_base:row_base + N_WINDOWS] = full_scores
                embeddings[row_base:row_base + N_WINDOWS] = full_emb
            else:
                scores[row_base:row_base + N_WINDOWS] = computed_scores
                embeddings[row_base:row_base + N_WINDOWS] = batch_emb

        write_file += batch_n
        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours,
    })
    return meta_df, scores, embeddings


## Cache Management
Load pre-computed Perch outputs or compute from scratch.

In [ ]:
def resolve_full_cache_paths():
    candidates = []
    for d in RUN_CFG["cache_input_dirs"]:
        if d.exists():
            candidates.append((d / "full_perch_meta.parquet", d / "full_perch_arrays.npz"))
    candidates.append((
        RUN_CFG["full_cache_work_dir"] / "full_perch_meta.parquet",
        RUN_CFG["full_cache_work_dir"] / "full_perch_arrays.npz",
    ))
    for meta_path, npz_path in candidates:
        if meta_path.exists() and npz_path.exists():
            return meta_path, npz_path
    return None, None

cache_meta, cache_npz = resolve_full_cache_paths()

if cache_meta is not None and cache_npz is not None:
    print("Loading cached Perch outputs from:", cache_meta)
    meta_full = pd.read_parquet(cache_meta)
    arr = np.load(cache_npz)
    scores_full_raw = arr["scores_full_raw"].astype(np.float32)
    emb_full = arr["emb_full"].astype(np.float32)
else:
    if MODE == "submit" and RUN_CFG["require_full_cache_in_submit"]:
        raise FileNotFoundError(
            "Submit mode requires cached Perch outputs. "
            "Attach the cache dataset or run in optimize mode first."
        )
    print("No cache found. Running Perch on trusted full files...")
    full_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_with_embeddings(
        full_paths, batch_files=RUN_CFG["batch_files"], verbose=True,
        proxy_reduce=RUN_CFG["proxy_reduce"],
    )
    out_meta = RUN_CFG["full_cache_work_dir"] / "full_perch_meta.parquet"
    out_npz = RUN_CFG["full_cache_work_dir"] / "full_perch_arrays.npz"
    meta_full.to_parquet(out_meta, index=False)
    np.savez_compressed(out_npz, scores_full_raw=scores_full_raw, emb_full=emb_full)
    print("Saved cache to:", out_meta)

# Align truth labels to cached order
full_truth_aligned = full_truth.set_index("row_id").loc[meta_full["row_id"]].reset_index()
Y_FULL = Y_SC[full_truth_aligned["index"].to_numpy()]

assert np.all(full_truth_aligned["filename"].values == meta_full["filename"].values)
assert np.all(full_truth_aligned["row_id"].values == meta_full["row_id"].values)

print("meta_full:", meta_full.shape)
print("scores_full_raw:", scores_full_raw.shape)
print("emb_full:", emb_full.shape)
print("Y_FULL:", Y_FULL.shape)

## Prior Tables
Bayesian site/hour/site-hour prior probability tables from labeled data.

In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    """Build hierarchical prior tables from labeled soundscape data."""
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = prior_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_prior[mask].mean(axis=0)

    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), dtype=np.float32)

    return {
        "global_p": global_p, "site_to_i": site_to_i, "site_n": site_n, "site_p": site_p,
        "hour_to_i": hour_to_i, "hour_n": hour_n, "hour_p": hour_p,
        "sh_to_i": sh_to_i, "sh_n": sh_n, "sh_p": sh_p,
    }

def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    """Compute logits from prior tables with Bayesian blending."""
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables["site_to_i"].get(str(s), -1) for s in sites), dtype=np.int32, count=n)
    hour_idx = np.fromiter((tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours), dtype=np.int32, count=n)
    sh_idx = np.fromiter((tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)), dtype=np.int32, count=n)

    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)

def fuse_scores_with_tables(base_scores, sites, hours, tables,
                            lambda_event=0.4, lambda_texture=1.0,
                            lambda_proxy_texture=0.8,
                            smooth_texture=0.35, smooth_event=0.15):
    """Fuse base Perch logits with site/hour priors and apply temporal smoothing."""
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = lambda_event * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture)
    scores = smooth_events_fixed12(scores, idx_active_event, alpha=smooth_event)
    return scores.astype(np.float32, copy=False), prior

# Fit prior tables on all labeled data
final_prior_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)
print("Built prior tables.")

## MHTAN: Multi-Head Temporal Attention Network

A transformer-based temporal classifier over Perch v2 embeddings.  
Key design choices:
- **Pre-norm residual blocks** for stable training on small data
- **Learnable positional encoding** for 12-window temporal structure
- **Perch residual gate** — architecture-level knowledge distillation via learnable sigmoid blend

In [ ]:
class TemporalAttentionBlock(nn.Module):
    """Pre-norm transformer block: multi-head self-attention + FFN."""

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


class MHTAN(nn.Module):
    """Multi-Head Temporal Attention Network for bioacoustic classification.

    Architecture:
    1. Linear projection: Perch embeddings (1536 -> d_model)
    2. Learnable positional encoding (12 windows)
    3. N stacked TemporalAttentionBlocks
    4. Per-window classification head
    5. Learnable Perch logit residual connection
    """

    def __init__(self, d_input=1536, d_model=128, n_heads=4, n_layers=2,
                 d_ff_mult=4, n_classes=234, n_windows=12, dropout=0.15,
                 use_perch_residual=True):
        super().__init__()
        self.n_classes = n_classes
        self.use_perch_residual = use_perch_residual

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        d_ff = d_model * d_ff_mult
        self.blocks = nn.ModuleList([
            TemporalAttentionBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.cls_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_classes),
        )

        if use_perch_residual:
            self.perch_gate = nn.Parameter(torch.tensor(0.3))

    def forward(self, emb, perch_logits=None):
        """Forward pass.
        Args:
            emb: (B, T, 1536) Perch embeddings per file
            perch_logits: (B, T, n_classes) Perch mapped logits (optional)
        Returns:
            (B, T, n_classes) per-window logits
        """
        x = self.input_proj(emb)
        x = x + self.pos_enc[:, :x.shape[1], :]

        for block in self.blocks:
            x = block(x)

        logits = self.cls_head(x)

        if self.use_perch_residual and perch_logits is not None:
            gate = torch.sigmoid(self.perch_gate)
            logits = (1.0 - gate) * logits + gate * perch_logits

        return logits

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print("MHTAN defined.")

## neuropt Search Space & Training Function

Define the hyperparameter search space and a training function that neuropt's LLM can optimize.  
The LLM reads per-epoch training curves and proposes better configurations.

In [ ]:
def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
    """Reshape flat (n_windows*n_files, ...) to (n_files, n_windows, ...)."""
    filenames = meta_df["filename"].to_numpy()
    unique_files = []
    seen = set()
    for f in filenames:
        if f not in seen:
            unique_files.append(f)
            seen.add(f)
    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows
    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files


def make_train_fn(emb_full, scores_full_raw, Y_FULL, meta_full):
    """Create a neuropt-compatible train_fn closure over the training data."""

    # Pre-reshape to file-level arrays (done once)
    emb_files, file_list = reshape_to_files(emb_full, meta_full)
    logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
    labels_files, _ = reshape_to_files(Y_FULL, meta_full)

    n_files = len(file_list)
    groups = np.array([meta_full["filename"].to_numpy()[i * N_WINDOWS] for i in range(n_files)])

    def train_fn(config):
        """Train MHTAN with given config, return score + curves for neuropt."""
        d_model = int(config.get("d_model", 128))
        n_heads = int(config.get("n_heads", 4))
        n_layers = int(config.get("n_layers", 2))
        d_ff_mult = int(config.get("d_ff_mult", 4))
        dropout = float(config.get("dropout", 0.15))
        lr = float(config.get("lr", 2e-3))
        weight_decay = float(config.get("weight_decay", 1e-3))
        n_epochs = int(config.get("n_epochs", 80))
        use_perch_residual = bool(config.get("use_perch_residual", True))
        pos_weight_cap = float(config.get("pos_weight_cap", 30.0))
        label_smoothing = float(config.get("label_smoothing", 0.0))

        # Ensure d_model is divisible by n_heads
        d_model = max(n_heads, (d_model // n_heads) * n_heads)

        # Single-fold split for speed during optimization
        gkf = GroupKFold(n_splits=5)
        for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(np.arange(n_files), groups=groups)):
            if fold_idx >= 1:
                break

        emb_tr = torch.tensor(emb_files[tr_idx], dtype=torch.float32)
        logits_tr = torch.tensor(logits_files[tr_idx], dtype=torch.float32)
        labels_tr = torch.tensor(labels_files[tr_idx], dtype=torch.float32)
        emb_va = torch.tensor(emb_files[va_idx], dtype=torch.float32)
        logits_va = torch.tensor(logits_files[va_idx], dtype=torch.float32)
        labels_va = torch.tensor(labels_files[va_idx], dtype=torch.float32)

        if label_smoothing > 0:
            labels_tr = labels_tr * (1 - label_smoothing) + label_smoothing / N_CLASSES

        model = MHTAN(
            d_input=1536, d_model=d_model, n_heads=n_heads,
            n_layers=n_layers, d_ff_mult=d_ff_mult,
            n_classes=N_CLASSES, n_windows=N_WINDOWS,
            dropout=dropout, use_perch_residual=use_perch_residual,
        )

        # Class weights for imbalanced data
        pos_count = labels_files[tr_idx].reshape(-1, N_CLASSES).sum(axis=0)
        total = labels_files[tr_idx].reshape(-1, N_CLASSES).shape[0]
        pw = np.clip((total - pos_count) / (pos_count + 1), 1.0, pos_weight_cap)
        pos_weight = torch.tensor(pw, dtype=torch.float32)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
            pct_start=0.1, anneal_strategy='cos',
        )

        train_losses, val_losses = [], []
        best_val_loss = float("inf")
        best_state = None

        for epoch in range(n_epochs):
            model.train()
            out = model(emb_tr, logits_tr)
            loss = F.binary_cross_entropy_with_logits(
                out, labels_tr, pos_weight=pos_weight[None, None, :]
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_losses.append(float(loss.item()))

            model.eval()
            with torch.no_grad():
                val_out = model(emb_va, logits_va)
                val_loss = F.binary_cross_entropy_with_logits(
                    val_out, labels_va, pos_weight=pos_weight[None, None, :]
                )
            val_losses.append(float(val_loss.item()))

            if val_loss.item() < best_val_loss:
                best_val_loss = val_loss.item()
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

        # Compute AUC with best model
        if best_state is not None:
            model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            val_pred = model(emb_va, logits_va).reshape(-1, N_CLASSES).numpy()
            val_true = labels_files[va_idx].reshape(-1, N_CLASSES)

        try:
            val_auc = macro_auc_skip_empty(val_true, val_pred)
        except Exception:
            val_auc = 0.0

        return {
            "score": float(val_auc),
            "train_losses": train_losses,
            "val_losses": val_losses,
            "val_auc": float(val_auc),
            "best_val_loss": float(best_val_loss),
            "n_params": model.count_parameters(),
        }

    return train_fn


train_fn = make_train_fn(emb_full, scores_full_raw, Y_FULL, meta_full)
print("train_fn ready.")

## neuropt Optimization

Run LLM-guided hyperparameter search.  
Claude analyzes per-epoch training/validation curves to detect overfitting, underfitting, and convergence issues, then proposes informed next experiments.

**Requires internet access** — only runs in `optimize` mode.

In [ ]:
if MODE == "optimize":
    from neuropt import ArchSearch, LogUniform, IntUniform, Categorical

    search_space = {
        # Architecture
        "d_model":            IntUniform(64, 256),
        "n_heads":            Categorical([2, 4, 8]),
        "n_layers":           IntUniform(1, 4),
        "d_ff_mult":          Categorical([2, 3, 4]),
        "dropout":            (0.05, 0.4),
        "use_perch_residual": Categorical([True, False]),

        # Training
        "lr":                 LogUniform(5e-4, 5e-2),
        "weight_decay":       LogUniform(1e-4, 1e-2),
        "n_epochs":           IntUniform(40, 120),
        "pos_weight_cap":     (10.0, 50.0),
        "label_smoothing":    (0.0, 0.1),
    }

    ml_context = """
BirdCLEF 2026 bioacoustic species classification:
- Input: Perch v2 embeddings (1536-dim) from 12 consecutive 5-second windows per 60s recording.
- Output: Multi-label classification across 234 species (birds, amphibians, mammals, reptiles, insects).
- Metric: Macro-averaged ROC-AUC (skipping classes with no true positives). Higher is better.
- Dataset: ~150 labeled soundscape files (~1800 windows), highly class-imbalanced (many rare species).
- Model: Transformer-style temporal attention network over Perch embeddings.
- The Perch residual gate (use_perch_residual) gives a strong baseline — the model should learn to
  improve on Perch, not fight it. Disabling it removes this safety net.
- Overfitting is the main risk given small dataset. Watch for val_loss diverging from train_loss.
- d_model must be divisible by n_heads (enforced in code, so propose compatible values).
- pos_weight_cap controls class imbalance handling: too high destabilizes training.
- label_smoothing helps prevent overconfident predictions on rare classes.
"""

    search = ArchSearch(
        train_fn=train_fn,
        search_space=search_space,
        backend="claude",
        log_path="/kaggle/working/neuropt_birdclef.jsonl",
        batch_size=3,
        ml_context=ml_context,
        minimize=False,  # Maximize AUC
    )

    print("Starting neuropt optimization...")
    print(f"Search space: {len(search_space)} parameters")
    print(f"Backend: claude | Batch size: 3 | Max evals: 20")
    print()

    search.run(max_evals=20)

    print(f"\nOptimization complete!")
    print(f"Best AUC: {search.best_score:.4f}")
    print(f"Best config: {json.dumps(search.best_config, indent=2, default=str)}")
    print(f"Total experiments: {search.total_experiments}")

    # Save best config
    with open("/kaggle/working/best_config.json", "w") as f:
        json.dump(search.best_config, f, indent=2, default=str)
    print("Saved best config to /kaggle/working/best_config.json")

else:
    print("Submit mode: skipping neuropt optimization, using FROZEN_BEST_CONFIG.")

## Select Config & Train Final Model
Use neuropt's best config (optimize mode) or frozen config (submit mode).  
Train the final model on ALL labeled data with early stopping.

In [ ]:
# Select config
if MODE == "optimize":
    config = search.best_config.copy()
    # Merge fusion params from frozen config (not searched by neuropt)
    for k in ["ensemble_weight_attn", "temperature", "lambda_event",
              "lambda_texture", "lambda_proxy_texture", "smooth_texture", "smooth_event"]:
        if k not in config:
            config[k] = FROZEN_BEST_CONFIG[k]
    print("Using neuropt best config.")
else:
    config = FROZEN_BEST_CONFIG.copy()
    print("Using frozen config.")

# Ensure d_model alignment
d_model = int(config["d_model"])
n_heads = int(config["n_heads"])
d_model = max(n_heads, (d_model // n_heads) * n_heads)
config["d_model"] = d_model

print("Final config:", json.dumps(config, indent=2, default=str))

# Reshape all training data to file-level
emb_files, file_list = reshape_to_files(emb_full, meta_full)
logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
labels_files, _ = reshape_to_files(Y_FULL, meta_full)

n_files = len(file_list)
n_val = max(1, int(n_files * 0.15))

perm = torch.randperm(n_files, generator=torch.Generator().manual_seed(42))
val_idx = perm[:n_val]
train_idx = perm[n_val:]

emb_tr = torch.tensor(emb_files[train_idx], dtype=torch.float32)
logits_tr = torch.tensor(logits_files[train_idx], dtype=torch.float32)
labels_tr = torch.tensor(labels_files[train_idx], dtype=torch.float32)
emb_va = torch.tensor(emb_files[val_idx], dtype=torch.float32)
logits_va = torch.tensor(logits_files[val_idx], dtype=torch.float32)
labels_va = torch.tensor(labels_files[val_idx], dtype=torch.float32)

label_smoothing = float(config.get("label_smoothing", 0.0))
if label_smoothing > 0:
    labels_tr = labels_tr * (1 - label_smoothing) + label_smoothing / N_CLASSES

# Build model
model = MHTAN(
    d_input=1536, d_model=d_model, n_heads=n_heads,
    n_layers=int(config["n_layers"]), d_ff_mult=int(config["d_ff_mult"]),
    n_classes=N_CLASSES, n_windows=N_WINDOWS,
    dropout=float(config["dropout"]),
    use_perch_residual=bool(config.get("use_perch_residual", True)),
)
print(f"MHTAN parameters: {model.count_parameters():,}")

# Class weights
pos_count = labels_files[train_idx].reshape(-1, N_CLASSES).sum(axis=0)
total = labels_files[train_idx].reshape(-1, N_CLASSES).shape[0]
pw = np.clip((total - pos_count) / (pos_count + 1), 1.0, float(config["pos_weight_cap"]))
pos_weight = torch.tensor(pw, dtype=torch.float32)

n_epochs = int(config["n_epochs"])
lr = float(config["lr"])

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=float(config["weight_decay"]))
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
    pct_start=0.1, anneal_strategy='cos',
)

# Training loop with early stopping
best_val_loss = float("inf")
best_state = None
patience = 20
wait = 0
train_history = {"train_loss": [], "val_loss": [], "val_auc": []}

t0 = time.time()
for epoch in range(n_epochs):
    model.train()
    out = model(emb_tr, logits_tr)
    loss = F.binary_cross_entropy_with_logits(out, labels_tr, pos_weight=pos_weight[None, None, :])

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        val_out = model(emb_va, logits_va)
        val_loss = F.binary_cross_entropy_with_logits(
            val_out, labels_va, pos_weight=pos_weight[None, None, :]
        )
        val_pred = val_out.reshape(-1, N_CLASSES).numpy()
        val_true = labels_files[val_idx].reshape(-1, N_CLASSES)
        try:
            val_auc = macro_auc_skip_empty(val_true, val_pred)
        except Exception:
            val_auc = 0.0

    train_history["train_loss"].append(loss.item())
    train_history["val_loss"].append(val_loss.item())
    train_history["val_auc"].append(val_auc)

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    if (epoch + 1) % 20 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:3d}: train={loss.item():.4f} val={val_loss.item():.4f} "
              f"auc={val_auc:.4f} lr={lr_now:.6f} wait={wait}")

    if wait >= patience:
        print(f"  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})")
        break

train_time = time.time() - t0

if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nTraining complete in {train_time:.1f}s")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Best val_auc: {max(train_history['val_auc']):.4f}")

if bool(config.get("use_perch_residual", True)):
    with torch.no_grad():
        gate_val = torch.sigmoid(model.perch_gate).item()
        print(f"Perch gate: {gate_val:.3f} (1=all Perch, 0=all MHTAN)")

## Test Inference
Run Perch on hidden test soundscapes.

In [ ]:
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))

if len(test_paths) == 0:
    print(f"Hidden test not mounted. Dry-run on first {RUN_CFG['dryrun_n_files']} train soundscapes.")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:RUN_CFG["dryrun_n_files"]]
else:
    print(f"Hidden test files: {len(test_paths)}")

_wall_elapsed = (time.time() - _WALL_START) / 60.0
print(f"Wall time before test inference: {_wall_elapsed:.1f} min")

# Subsample: process 6/12 windows, interpolate the rest — cuts inference ~50%
USE_SUBSAMPLE = (MODE == "submit")
if USE_SUBSAMPLE:
    print(f"Subsampling: processing 6/12 windows per file (interpolating rest)")

meta_test, scores_test_raw, emb_test = infer_perch_with_embeddings(
    test_paths, batch_files=RUN_CFG["batch_files"], verbose=True,
    proxy_reduce=RUN_CFG["proxy_reduce"], subsample=USE_SUBSAMPLE,
)

_wall_after = (time.time() - _WALL_START) / 60.0
_perch_time = _wall_after - _wall_elapsed
print(f"Test inference: {_perch_time:.1f} min for {len(test_paths)} files")
if len(test_paths) <= 50:
    est = (_perch_time / len(test_paths)) * 600 + _wall_elapsed
    print(f"Estimated total for 600 test files: {est:.1f} min / 90 min")
print(f"meta_test: {meta_test.shape} | emb_test: {emb_test.shape}")


## Score Fusion & Submission

Combine MHTAN temporal predictions with prior-fused Perch base scores.  
Apply temperature scaling and generate `submission.csv`.

In [ ]:
# Step 1: MHTAN inference on test
emb_test_files, test_file_list = reshape_to_files(emb_test, meta_test)
logits_test_files, _ = reshape_to_files(scores_test_raw, meta_test)

emb_test_t = torch.tensor(emb_test_files, dtype=torch.float32)
logits_test_t = torch.tensor(logits_test_files, dtype=torch.float32)

model.eval()
with torch.no_grad():
    mhtan_scores = model(emb_test_t, logits_test_t)
    mhtan_scores_flat = mhtan_scores.reshape(-1, N_CLASSES).numpy().astype(np.float32)

print(f"MHTAN test scores: {mhtan_scores_flat.shape}")

# Step 2: Prior-fused Perch base scores
test_base_scores, _ = fuse_scores_with_tables(
    scores_test_raw,
    sites=meta_test["site"].to_numpy(),
    hours=meta_test["hour_utc"].to_numpy(),
    tables=final_prior_tables,
    lambda_event=float(config.get("lambda_event", 0.4)),
    lambda_texture=float(config.get("lambda_texture", 1.0)),
    lambda_proxy_texture=float(config.get("lambda_proxy_texture", 0.8)),
    smooth_texture=float(config.get("smooth_texture", 0.35)),
    smooth_event=float(config.get("smooth_event", 0.15)),
)

# Step 3: Ensemble
W = float(config.get("ensemble_weight_attn", 0.5))
final_test_scores = (
    W * mhtan_scores_flat + (1.0 - W) * test_base_scores
).astype(np.float32)

print(f"Ensemble weights: MHTAN={W:.2f}, Perch+Priors={1-W:.2f}")

# Step 4: Temperature scaling + submission
TEMPERATURE = float(config.get("temperature", 1.10))
submission = pd.DataFrame(sigmoid(final_test_scores / TEMPERATURE), columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

expected_rows = len(test_paths) * N_WINDOWS
assert len(submission) == expected_rows, f"Expected {expected_rows}, got {len(submission)}"
assert submission.columns.tolist() == ["row_id"] + PRIMARY_LABELS
assert not submission.isna().any().any()

submission.to_csv("submission.csv", index=False)

print(f"\nSaved submission.csv")
print(f"Shape: {submission.shape}")
print(f"Temperature: {TEMPERATURE}")
print(submission.iloc[:3, :8])

## Diagnostics

In [ ]:
_wall_total = (time.time() - _WALL_START) / 60.0
print(f"Total wall time: {_wall_total:.1f} min / 90 min")
print(f"Best val AUC: {max(train_history['val_auc']):.4f}")
print(f"Params: {model.count_parameters():,}")
print("Done.")
